# Chest X-ray Embedding Extraction — Hybrid Chunk-wise Version

This notebook extracts frozen 1024-dimensional embeddings from the pretrained TorchXRayVision DenseNet121 model.

### Data strategy
Google Drive is used only as persistent storage. Images are processed in **2,000-image chunks**:

1. Read image paths from Google Drive.
2. Copy one chunk to local Colab storage (`/content`).
3. Load that local chunk with a PyTorch DataLoader.
4. Extract 1024-dimensional embeddings on the GPU.
5. Save one compressed `.npz` checkpoint to Google Drive.
6. Delete the local chunk.
7. Continue with the next chunk.

This avoids Google Drive API per-image requests and avoids repeatedly training/reading the entire dataset directly from mounted Drive.

**Expected split sizes**
- Train: 14,000
- Validation: 3,000
- Test: 3,000

**GPU required:** NVIDIA L4 or another CUDA GPU.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip install -q torchxrayvision


In [17]:
import os
import shutil
import gc
from pathlib import Path
import math
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchxrayvision as xrv

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


In [4]:
# Runtime and GPU setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type != "cuda":
    raise RuntimeError(
        "GPU is required for embedding extraction. "
        "Go to Runtime > Change runtime type and select a GPU."
    )

# Input size and batch size stay fixed, so cuDNN can benchmark
# the best convolution algorithms for this workload.
torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130
Device: cuda
GPU: NVIDIA L4


In [5]:
# Dataset locations

MY_DRIVE = "/content/drive/MyDrive"

DATA_ROOT = os.path.join(
    MY_DRIVE,
    "NIH_ChestXray",
    "Training_Data"
)

SPLIT_PATHS = {
    "train": os.path.join(DATA_ROOT, "train"),
    "validation": os.path.join(DATA_ROOT, "validation"),
    "test": os.path.join(DATA_ROOT, "test"),
}

print("NIH_ChestXray exists:",
      os.path.exists(os.path.join(MY_DRIVE, "NIH_ChestXray")))

print("Training_Data exists:", os.path.isdir(DATA_ROOT))

for split, path in SPLIT_PATHS.items():
    print(f"{split}: {path} | exists = {os.path.isdir(path)}")


NIH_ChestXray exists: True
Training_Data exists: True
train: /content/drive/MyDrive/NIH_ChestXray/Training_Data/train | exists = True
validation: /content/drive/MyDrive/NIH_ChestXray/Training_Data/validation | exists = True
test: /content/drive/MyDrive/NIH_ChestXray/Training_Data/test | exists = True


In [6]:
# Chunk-wise processing configuration

CHUNK_SIZE = 2000
BATCH_SIZE = 64

LOCAL_ROOT = "/content/chest_xray_chunks"

CHECKPOINT_ROOT = os.path.join(
    MY_DRIVE,
    "Reliable-Chest-X-Ray-AI",
    "embeddings_checkpoints"
)

SPLIT_OUTPUT_DIRS = {
    "train": os.path.join(CHECKPOINT_ROOT, "train"),
    "validation": os.path.join(CHECKPOINT_ROOT, "validation"),
    "test": os.path.join(CHECKPOINT_ROOT, "test"),
}

os.makedirs(LOCAL_ROOT, exist_ok=True)

for directory in SPLIT_OUTPUT_DIRS.values():
    os.makedirs(directory, exist_ok=True)

print("Chunk size:", CHUNK_SIZE)
print("Batch size:", BATCH_SIZE)
print("Local root:", LOCAL_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)


Chunk size: 2000
Batch size: 64
Local root: /content/chest_xray_chunks
Checkpoint root: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints


In [7]:
# Count images on Google Drive

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")

def count_local_images(folder):
    count = 0

    for root, _, filenames in os.walk(folder):
        count += sum(
            filename.lower().endswith(IMAGE_EXTENSIONS)
            for filename in filenames
        )

    return count


for split, path in SPLIT_PATHS.items():
    count = count_local_images(path)
    print(f"{split}: {count} images")


train: 14000 images
validation: 3000 images
test: 3000 images


In [8]:
# Collect image paths

def get_image_paths(folder):
    paths = []

    for root, _, filenames in os.walk(folder):
        for filename in filenames:
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                paths.append(os.path.join(root, filename))

    paths.sort()
    return paths


train_paths = get_image_paths(SPLIT_PATHS["train"])

assert len(train_paths) == 14000, (
    f"Expected 14000 train images, found {len(train_paths)}"
)

print(f"Train images found: {len(train_paths)}")
print("First image:", train_paths[0])
print("Last image:", train_paths[-1])


Train images found: 14000
First image: /content/drive/MyDrive/NIH_ChestXray/Training_Data/train/00000003_003.png
Last image: /content/drive/MyDrive/NIH_ChestXray/Training_Data/train/00030786_007.png


In [9]:
# Dataset used for one local chunk

class XrayDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

        self.transform = torchvision.transforms.Compose([
            xrv.datasets.XRayCenterCrop(),
            xrv.datasets.XRayResizer(224)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]

        image = Image.open(path).convert("L")
        image = np.asarray(image, dtype=np.float32)

        # TorchXRayVision normalization
        image = xrv.datasets.normalize(image, 255)

        # Add channel dimension: [H, W] -> [1, H, W]
        image = image[None, :, :]

        # TorchXRayVision preprocessing
        image = self.transform(image)

        image = torch.from_numpy(image).float()

        return image, os.path.basename(path)


In [10]:
# Pretrained frozen model

model = xrv.models.DenseNet(
    weights="densenet121-res224-chex"
)

model = model.to(device)
model.eval()

for param in model.parameters():
    param.requires_grad_(False)

print("Model loaded and frozen.")


If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
Model loaded and frozen.


In [11]:
# Utility: copy one chunk from Google Drive to local Colab storage

def prepare_local_chunk(paths, split, chunk_idx):
    chunk_dir = os.path.join(
        LOCAL_ROOT,
        split,
        f"chunk_{chunk_idx:03d}"
    )

    if os.path.isdir(chunk_dir):
        shutil.rmtree(chunk_dir)

    os.makedirs(chunk_dir, exist_ok=True)

    print(
        f"Copying {len(paths)} images for "
        f"{split} chunk {chunk_idx:03d}..."
    )

    for src in tqdm(paths, desc=f"Copying {split} chunk {chunk_idx:03d}"):
        filename = os.path.basename(src)
        dst = os.path.join(chunk_dir, filename)

        shutil.copy2(src, dst)

    print(f"Local chunk ready: {chunk_dir}")

    return chunk_dir


In [12]:
# Utility: build a DataLoader for one local chunk

def build_chunk_loader(chunk_dir):
    paths = get_image_paths(chunk_dir)

    dataset = XrayDataset(paths)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    return dataset, loader


In [13]:
# Utility: save checkpoint atomically

def save_checkpoint(path, names, embeddings):
    temp_path = path + ".tmp.npz"

    np.savez_compressed(
        temp_path,
        image_names=np.asarray(names),
        embeddings=embeddings
    )

    # The final file appears only after the temporary file is complete.
    os.replace(temp_path, path)


In [14]:
# One-batch GPU sanity check using the first 64 train images

sanity_paths = train_paths[:BATCH_SIZE]

sanity_dir = prepare_local_chunk(
    sanity_paths,
    split="sanity",
    chunk_idx=0
)

sanity_dataset, sanity_loader = build_chunk_loader(sanity_dir)

images, names = next(iter(sanity_loader))

with torch.inference_mode():
    images = images.to(device, non_blocking=True)

    features = model.features(images)
    features = F.relu(features, inplace=True)

    embeddings = F.adaptive_avg_pool2d(
        features,
        (1, 1)
    ).flatten(1)

print("Images shape:", tuple(images.shape))
print("Embeddings shape:", tuple(embeddings.shape))

assert embeddings.shape[0] == images.shape[0]
assert embeddings.shape[1] == 1024

print("GPU sanity check passed.")

# Remove sanity chunk immediately.
shutil.rmtree(sanity_dir)
print("Sanity chunk deleted.")


Copying 64 images for sanity chunk 000...


Copying sanity chunk 000:   0%|          | 0/64 [00:00<?, ?it/s]

Local chunk ready: /content/chest_xray_chunks/sanity/chunk_000
Images shape: (64, 1, 224, 224)
Embeddings shape: (64, 1024)
GPU sanity check passed.
Sanity chunk deleted.


In [15]:
# Extract embeddings chunk by chunk

def extract_split_in_chunks(split, all_paths, expected_count=None):
    if expected_count is not None:
        assert len(all_paths) == expected_count, (
            f"{split}: expected {expected_count} images, "
            f"found {len(all_paths)}"
        )

    output_dir = SPLIT_OUTPUT_DIRS[split]
    os.makedirs(output_dir, exist_ok=True)

    total_images = len(all_paths)
    num_chunks = math.ceil(total_images / CHUNK_SIZE)

    print(f"\n===== {split.upper()} =====")
    print(f"Images: {total_images}")
    print(f"Chunks: {num_chunks}")
    print(f"Chunk size: {CHUNK_SIZE}")

    for chunk_idx in range(num_chunks):
        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, total_images)

        chunk_paths = all_paths[start:end]

        checkpoint_path = os.path.join(
            output_dir,
            f"chunk_{chunk_idx:03d}.npz"
        )

        # Resume support: do not copy/process a chunk that
        # already has a persistent checkpoint on Drive.
        if os.path.exists(checkpoint_path):
            print(
                f"Skipping {split} chunk {chunk_idx:03d}: "
                f"checkpoint already exists."
            )
            continue

        local_chunk_dir = prepare_local_chunk(
            chunk_paths,
            split,
            chunk_idx
        )

        try:
            dataset, loader = build_chunk_loader(local_chunk_dir)

            assert len(dataset) == len(chunk_paths), (
                f"{split} chunk {chunk_idx}: "
                f"expected {len(chunk_paths)} images, "
                f"found {len(dataset)}"
            )

            all_names = []
            all_embeddings = []

            progress = tqdm(
                loader,
                total=len(loader),
                desc=f"Embedding {split} chunk {chunk_idx:03d}"
            )

            for images, names in progress:
                with torch.inference_mode():
                    images = images.to(
                        device,
                        non_blocking=True
                    )

                    features = model.features(images)
                    features = F.relu(features, inplace=True)

                    embeddings = F.adaptive_avg_pool2d(
                        features,
                        (1, 1)
                    ).flatten(1)

                embeddings = embeddings.cpu().numpy()

                assert embeddings.shape[0] == len(names)
                assert embeddings.shape[1] == 1024

                all_names.extend(names)
                all_embeddings.append(embeddings)

            chunk_embeddings = np.concatenate(
                all_embeddings,
                axis=0
            )

            assert len(all_names) == len(chunk_paths)
            assert chunk_embeddings.shape == (
                len(chunk_paths),
                1024
            )

            save_checkpoint(
                checkpoint_path,
                all_names,
                chunk_embeddings
            )

            print(
                f"Saved checkpoint: {checkpoint_path}"
            )
            print(
                f"Saved embeddings shape: "
                f"{chunk_embeddings.shape}"
            )

        finally:
            # Always delete the local chunk, even if an error occurs.
            if os.path.isdir(local_chunk_dir):
                shutil.rmtree(local_chunk_dir)

            gc.collect()
            torch.cuda.empty_cache()

            print(
                f"Deleted local {split} chunk "
                f"{chunk_idx:03d}."
            )

    print(f"\nFinished {split}.")


## Train

The train paths are already collected above.

Running the next cell processes **14,000 images in 7 chunks**:

- chunk 000 → images 0–1,999
- chunk 001 → images 2,000–3,999
- ...
- chunk 006 → images 12,000–13,999

Each completed chunk is saved to Google Drive before its local copy is deleted.


In [18]:
extract_split_in_chunks(
    split="train",
    all_paths=train_paths,
    expected_count=14000
)



===== TRAIN =====
Images: 14000
Chunks: 7
Chunk size: 2000
Copying 2000 images for train chunk 000...


Copying train chunk 000:   0%|          | 0/2000 [00:00<?, ?it/s]

Local chunk ready: /content/chest_xray_chunks/train/chunk_000


Embedding train chunk 000:   0%|          | 0/32 [00:00<?, ?it/s]

Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/train/chunk_000.npz
Saved embeddings shape: (2000, 1024)
Deleted local train chunk 000.
Copying 2000 images for train chunk 001...


Copying train chunk 001:   0%|          | 0/2000 [00:00<?, ?it/s]

... (19 outputs left out)

Copying train chunk 006:   0%|          | 0/2000 [00:00<?, ?it/s]

Local chunk ready: /content/chest_xray_chunks/train/chunk_006


Embedding train chunk 006:   0%|          | 0/32 [00:00<?, ?it/s]

Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/train/chunk_006.npz
Saved embeddings shape: (2000, 1024)
Deleted local train chunk 006.

Finished train.


In [19]:
# Verify train checkpoints

def verify_chunk_checkpoints(split, expected_count):
    output_dir = SPLIT_OUTPUT_DIRS[split]

    total_chunks = math.ceil(
        expected_count / CHUNK_SIZE
    )

    files = sorted(
        f for f in os.listdir(output_dir)
        if f.endswith(".npz")
        and f.startswith("chunk_")
    )

    print(f"{split}: {len(files)} checkpoint files found")
    print(f"Expected: {total_chunks}")

    assert len(files) == total_chunks, (
        f"{split}: expected {total_chunks} checkpoints, "
        f"found {len(files)}"
    )

    total_embeddings = 0
    all_names = []

    for filename in files:
        path = os.path.join(output_dir, filename)
        data = np.load(path)

        names = data["image_names"]
        embeddings = data["embeddings"]

        print(
            f"{filename}: "
            f"names={names.shape}, "
            f"embeddings={embeddings.shape}"
        )

        assert embeddings.ndim == 2
        assert embeddings.shape[1] == 1024
        assert embeddings.shape[0] == len(names)

        total_embeddings += embeddings.shape[0]
        all_names.extend(names.tolist())

    assert total_embeddings == expected_count, (
        f"{split}: expected {expected_count} total embeddings, "
        f"found {total_embeddings}"
    )

    assert len(set(all_names)) == expected_count, (
        f"{split}: duplicate image names detected."
    )

    print(
        f"{split}: checkpoint verification passed. "
        f"Total embeddings = {total_embeddings}"
    )


In [20]:
verify_chunk_checkpoints("train", 14000)


train: 7 checkpoint files found
Expected: 7
chunk_000.npz: names=(2000,), embeddings=(2000, 1024)
chunk_001.npz: names=(2000,), embeddings=(2000, 1024)
chunk_002.npz: names=(2000,), embeddings=(2000, 1024)
chunk_003.npz: names=(2000,), embeddings=(2000, 1024)
chunk_004.npz: names=(2000,), embeddings=(2000, 1024)
chunk_005.npz: names=(2000,), embeddings=(2000, 1024)
chunk_006.npz: names=(2000,), embeddings=(2000, 1024)
train: checkpoint verification passed. Total embeddings = 14000


In [23]:
import os
import shutil

# ZIP files on Google Drive
DATA_ROOT = "/content/drive/MyDrive/NIH_ChestXray/Training_Data"

# Local Colab directory
LOCAL_ROOT = "/content/chest_xray_data"

os.makedirs(LOCAL_ROOT, exist_ok=True)

for split in ["validation", "test"]:

    zip_path = os.path.join(DATA_ROOT, f"{split}.zip")
    local_zip = os.path.join(LOCAL_ROOT, f"{split}.zip")
    extract_path = os.path.join(LOCAL_ROOT, split)

    print(f"\n{'='*50}")
    print(f"Processing {split}.zip")
    print(f"{'='*50}")

    # Copy ZIP: Google Drive → Colab local storage
    print("Copying ZIP to Colab...")
    shutil.copy2(zip_path, local_zip)

    # Extract locally
    print("Extracting...")
    shutil.unpack_archive(local_zip, extract_path)

    # Delete local ZIP to save space
    os.remove(local_zip)

    # Count images
    image_count = sum(
        1
        for root, _, files in os.walk(extract_path)
        for f in files
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    )

    print(f"✓ {split} ready")
    print(f"✓ Images: {image_count}")
    print(f"✓ Local path: {extract_path}")

print("\n✅ Validation + Test extracted successfully!")


Processing validation.zip
Copying ZIP to Colab...
Extracting...
✓ validation ready
✓ Images: 3000
✓ Local path: /content/chest_xray_data/validation

Processing test.zip
Copying ZIP to Colab...
Extracting...
✓ test ready
✓ Images: 3000
✓ Local path: /content/chest_xray_data/test

✅ Validation + Test extracted successfully!


In [24]:
# Extract embeddings from locally extracted data

def extract_split_local(split, all_paths, expected_count=None):

    if expected_count is not None:
        assert len(all_paths) == expected_count, (
            f"{split}: expected {expected_count} images, "
            f"found {len(all_paths)}"
        )

    output_dir = SPLIT_OUTPUT_DIRS[split]
    os.makedirs(output_dir, exist_ok=True)

    total_images = len(all_paths)
    num_chunks = math.ceil(total_images / CHUNK_SIZE)

    print(f"\n===== {split.upper()} =====")
    print(f"Images: {total_images}")
    print(f"Chunks: {num_chunks}")
    print(f"Chunk size: {CHUNK_SIZE}")

    for chunk_idx in range(num_chunks):

        start = chunk_idx * CHUNK_SIZE
        end = min(start + CHUNK_SIZE, total_images)

        chunk_paths = all_paths[start:end]

        checkpoint_path = os.path.join(
            output_dir,
            f"chunk_{chunk_idx:03d}.npz"
        )

        # Resume support
        if os.path.exists(checkpoint_path):
            print(
                f"Skipping {split} chunk {chunk_idx:03d}: "
                f"checkpoint already exists."
            )
            continue

        print(
            f"\nProcessing {split} chunk "
            f"{chunk_idx:03d} "
            f"({start}:{end})"
        )

        dataset = XrayDataset(chunk_paths)

        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            persistent_workers=True
        )

        all_names = []
        all_embeddings = []

        progress = tqdm(
            loader,
            total=len(loader),
            desc=f"Embedding {split} chunk {chunk_idx:03d}"
        )

        for images, names in progress:

            with torch.inference_mode():

                images = images.to(
                    device,
                    non_blocking=True
                )

                features = model.features(images)

                features = F.relu(
                    features,
                    inplace=True
                )

                embeddings = F.adaptive_avg_pool2d(
                    features,
                    (1, 1)
                ).flatten(1)

            embeddings = embeddings.cpu().numpy()

            assert embeddings.shape[0] == len(names)
            assert embeddings.shape[1] == 1024

            all_names.extend(names)
            all_embeddings.append(embeddings)

        chunk_embeddings = np.concatenate(
            all_embeddings,
            axis=0
        )

        assert len(all_names) == len(chunk_paths)

        assert chunk_embeddings.shape == (
            len(chunk_paths),
            1024
        )

        # Save checkpoint directly to Google Drive
        save_checkpoint(
            checkpoint_path,
            all_names,
            chunk_embeddings
        )

        print(
            f"✓ Saved checkpoint: {checkpoint_path}"
        )

        print(
            f"✓ Embeddings shape: "
            f"{chunk_embeddings.shape}"
        )

        # Free memory
        del dataset
        del loader
        del all_names
        del all_embeddings
        del chunk_embeddings

        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n✓ Finished {split}.")

In [25]:
validation_paths = get_image_paths(
    "/content/chest_xray_data/validation"
)

print("Validation images:", len(validation_paths))
test_paths = get_image_paths(
    "/content/chest_xray_data/test"
)

print("Test images:", len(test_paths))

Validation images: 3000
Test images: 3000


In [26]:
extract_split_local(
    split="validation",
    all_paths=validation_paths,
    expected_count=3000
)


===== VALIDATION =====
Images: 3000
Chunks: 2
Chunk size: 2000

Processing validation chunk 000 (0:2000)


Embedding validation chunk 000:   0%|          | 0/32 [00:00<?, ?it/s]

✓ Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/validation/chunk_000.npz
✓ Embeddings shape: (2000, 1024)

Processing validation chunk 001 (2000:3000)


Embedding validation chunk 001:   0%|          | 0/16 [00:00<?, ?it/s]

✓ Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/validation/chunk_001.npz
✓ Embeddings shape: (1000, 1024)

✓ Finished validation.


In [27]:
extract_split_local(
    split="test",
    all_paths=test_paths,
    expected_count=3000
)


===== TEST =====
Images: 3000
Chunks: 2
Chunk size: 2000

Processing test chunk 000 (0:2000)


Embedding test chunk 000:   0%|          | 0/32 [00:00<?, ?it/s]

✓ Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/test/chunk_000.npz
✓ Embeddings shape: (2000, 1024)

Processing test chunk 001 (2000:3000)


Embedding test chunk 001:   0%|          | 0/16 [00:00<?, ?it/s]

✓ Saved checkpoint: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/test/chunk_001.npz
✓ Embeddings shape: (1000, 1024)

✓ Finished test.
